In [ ]:
from pathlib import Path
import sys
import importlib

import rqdatac

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
LIB_DIR = PROJECT_ROOT / "lib"
UTIL_FILE = LIB_DIR / "rqalpha_strategy_utils.py"

assert UTIL_FILE.exists(), (
    f"尚未找到工具文件：{UTIL_FILE}\n"
    "请先重新运行完整的 %%writefile lib/rqalpha_strategy_utils.py 单元。"
)

if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

rqdatac.init()

import rqalpha_strategy_utils as rau


print("RQAlpha 工具库已加载：", rau.__file__)
import json
import matplotlib.pyplot as plt

In [ ]:
import importlib
import rqalpha_strategy_utils as rau

importlib.reload(rau)
print("已重新加载：", rau.__file__)

In [ ]:
%%writefile lib/rqalpha_strategy_utils.py
from pathlib import Path

import pandas as pd
from rqalpha.api import order_target_percent
from rqalpha_plus import run_func


def _normalize_target_weights(target_weights, tolerance=1e-10):
    """将日期×资产目标权重表标准化，并转为 RQAlpha 可用的日期映射。"""
    if not isinstance(target_weights, pd.DataFrame):
        raise TypeError("target_weights 必须是 DataFrame。")

    panel = target_weights.copy()
    panel.index = pd.to_datetime(panel.index).normalize()
    panel = panel[~panel.index.duplicated(keep="last")].sort_index()
    panel = panel.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    panel.columns = panel.columns.astype(str)

    panel = panel.loc[:, panel.abs().sum(axis=0) > tolerance]
    panel = panel.mask(panel.abs() <= tolerance, 0.0)

    if (panel < -tolerance).any().any():
        raise ValueError("目标权重不能为负；本函数仅用于多头或纯现金策略。")

    if (panel.sum(axis=1) > 1.0 + tolerance).any():
        raise ValueError("存在单日目标总权重超过 100% 的情况。")

    schedule = {
        date.date(): {
            stock: float(weight)
            for stock, weight in panel.loc[date].items()
            if weight > tolerance
        }
        for date in panel.index
    }
    return panel, schedule


def run_target_weight_backtest(
    target_weights,
    *,
    initial_cash,
    benchmark,
    analyser_output_file,
    plot_output_path,
    matching_type="current_bar",
    min_commission=5.0,
    stock_commission_multiplier=0.375,
    tax_multiplier=1.0,
    price_limit=True,
    volume_limit=True,
    volume_percent=0.05,
    show_progress=True,
):
    """
    使用 RQAlpha 按目标权重调仓。

    每个目标权重日期在开盘集合竞价处理：
    先卖出不在目标中的旧持仓，再调已有股票，最后买入新股票。
    因涨跌停、停牌或成交量限制而未成交的委托会在下一交易日重试。
    """
    panel, targets_by_date = _normalize_target_weights(target_weights)

    analyser_output_file = Path(analyser_output_file)
    plot_output_path = Path(plot_output_path)
    analyser_output_file.parent.mkdir(parents=True, exist_ok=True)
    plot_output_path.mkdir(parents=True, exist_ok=True)

    start_date = panel.index.min().strftime("%Y-%m-%d")
    end_date = panel.index.max().strftime("%Y-%m-%d")

    def init(context):
        context.targets_by_date = targets_by_date
        context.order_queue = []
        context.pending_queue = []

    def before_trading(context):
        today = pd.Timestamp(context.now).date()

        if today in context.targets_by_date:
            target = context.targets_by_date[today]
            held = list(context.portfolio.positions.keys())

            sell_first = [stock for stock in held if target.get(stock, 0.0) == 0.0]
            held_targets = [stock for stock in held if stock not in sell_first]
            new_targets = [stock for stock in target if stock not in held]

            context.order_queue = list(
                dict.fromkeys(sell_first + held_targets + new_targets)
            )
            context.pending_queue = []
            context.current_target = target
        elif context.pending_queue:
            context.order_queue = list(context.pending_queue)

    def open_auction(context, bar_dict):
        if not context.order_queue:
            return

        failed_orders = []
        for stock in context.order_queue:
            target_weight = context.current_target.get(stock, 0.0)
            order = order_target_percent(stock, target_weight)
            if order is None:
                failed_orders.append(stock)

        context.pending_queue = failed_orders
        context.order_queue = []

    def after_trading(context):
        pass

    config = {
        "base": {
            "start_date": start_date,
            "end_date": end_date,
            "frequency": "1d",
            "accounts": {"STOCK": float(initial_cash)},
        },
        "mod": {
            "sys_accounts": {
                "enabled": True,
                "dividend_reinvestment": True,
            },
            "sys_analyser": {
                "enabled": True,
                "benchmark": benchmark,
                "plot": True,
                "output_file": str(analyser_output_file),
                "plot_save_file": str(plot_output_path),
            },
            "sys_risk": {
                "enabled": True,
                "validate_is_trading": True,
            },
            "sys_simulation": {
                "enabled": True,
                "signal": True,
                "matching_type": matching_type,
                "price_limit": price_limit,
                "liquidity_limit": False,
                "volume_limit": volume_limit,
                "volume_percent": volume_percent,
                "slippage_model": "PriceRatioSlippage",
                "slippage": 0,
            },
            "sys_transaction_cost": {
                "enabled": True,
                "cn_stock_min_commission": float(min_commission),
                "stock_commission_multiplier": float(stock_commission_multiplier),
                "tax_multiplier": float(tax_multiplier),
            },
            "sys_progress": {
                "enabled": True,
                "show": bool(show_progress),
            },
        },
    }

        result = run_func(
        config=config,
        init=init,
        before_trading=before_trading,
        open_auction=open_auction,
        after_trading=after_trading,
    )

    if result is None:
        raise RuntimeError(
            "RQAlpha 未返回有效结果；请检查上方 RQAlpha 策略日志。"
        )

    return result

In [ ]:
%%writefile lib/rqalpha_strategy_utils.py
from pathlib import Path

import pandas as pd
from rqalpha.api import order_target_percent
from rqalpha_plus import run_func


def _build_target_schedule(target_weights, tolerance=1e-10):
    """把日期×股票目标权重表转换为日期→目标权重字典。"""
    if not isinstance(target_weights, pd.DataFrame):
        raise TypeError("target_weights 必须是 DataFrame。")

    panel = target_weights.copy()
    panel.index = pd.to_datetime(panel.index).normalize()
    panel = panel[~panel.index.duplicated(keep="last")].sort_index()
    panel = panel.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    panel.columns = panel.columns.astype(str)
    panel = panel.loc[:, panel.abs().sum(axis=0) > tolerance]
    panel = panel.mask(panel.abs() <= tolerance, 0.0)

    if panel.empty:
        raise ValueError("目标权重表为空。")

    if (panel < -tolerance).any().any():
        raise ValueError("本回测函数仅支持非负目标权重。")

    if (panel.sum(axis=1) > 1.0 + tolerance).any():
        raise ValueError("存在单日目标总权重超过 100% 的情况。")

    schedule = {}
    for date in panel.index:
        schedule[date.date()] = {
            stock: float(weight)
            for stock, weight in panel.loc[date].items()
            if weight > tolerance
        }

    return panel, schedule


def run_target_weight_backtest(
    target_weights,
    initial_cash,
    benchmark,
    analyser_output_file,
    plot_output_path,
    matching_type="current_bar",
    min_commission=5.0,
    stock_commission_multiplier=0.375,
    tax_multiplier=1.0,
    price_limit=True,
    volume_limit=True,
    volume_percent=0.05,
    show_progress=True,
    progress_interval=20,
):
    """
    使用 RQAlpha 对任意日期×股票目标权重表执行开盘调仓回测。

    卖出非目标持仓优先；未成交委托在下一交易日重试。
    """
    panel, targets_by_date = _build_target_schedule(target_weights)

    analyser_output_file = Path(analyser_output_file)
    plot_output_path = Path(plot_output_path)
    analyser_output_file.parent.mkdir(parents=True, exist_ok=True)
    plot_output_path.mkdir(parents=True, exist_ok=True)

    def init(context):
        context.targets_by_date = targets_by_date
        context.current_target = {}
        context.order_queue = []
        context.pending_queue = []
        context.trading_day_count = 0

    def before_trading(context):
        context.trading_day_count += 1
        today = pd.Timestamp(context.now).date()

        if (
            context.trading_day_count == 1
            or context.trading_day_count % progress_interval == 0
        ):
            print(
                "[RQAlpha] 已处理第 {} 个交易日，当前日期：{}".format(
                    context.trading_day_count,
                    today,
                ),
                flush=True,
            )

        if today in context.targets_by_date:
            context.current_target = context.targets_by_date[today]
            held_stocks = list(context.portfolio.positions.keys())

            sell_first = [
                stock for stock in held_stocks
                if context.current_target.get(stock, 0.0) == 0.0
            ]
            held_targets = [
                stock for stock in held_stocks
                if stock not in sell_first
            ]
            new_targets = [
                stock for stock in context.current_target
                if stock not in held_stocks
            ]

            context.order_queue = list(
                dict.fromkeys(sell_first + held_targets + new_targets)
            )
            context.pending_queue = []

        elif context.pending_queue:
            context.order_queue = list(context.pending_queue)

    def open_auction(context, bar_dict):
        if not context.order_queue:
            return

        failed_orders = []

        for stock in context.order_queue:
            target_weight = context.current_target.get(stock, 0.0)
            order = order_target_percent(stock, target_weight)

            if order is None:
                failed_orders.append(stock)

        context.pending_queue = failed_orders
        context.order_queue = []

    def after_trading(context):
        pass

    config = {
        "base": {
            "start_date": panel.index.min().strftime("%Y-%m-%d"),
            "end_date": panel.index.max().strftime("%Y-%m-%d"),
            "frequency": "1d",
            "accounts": {"STOCK": float(initial_cash)},
        },
        "mod": {
            "sys_accounts": {
                "enabled": True,
                "dividend_reinvestment": True,
            },
            "sys_analyser": {
                "enabled": True,
                "benchmark": benchmark,
                "plot": True,
                "output_file": str(analyser_output_file),
                "plot_save_file": str(plot_output_path),
            },
            "sys_risk": {
                "enabled": True,
                "validate_is_trading": True,
            },
            "sys_simulation": {
                "enabled": True,
                "signal": True,
                "matching_type": matching_type,
                "price_limit": price_limit,
                "liquidity_limit": False,
                "volume_limit": volume_limit,
                "volume_percent": volume_percent,
                "slippage_model": "PriceRatioSlippage",
                "slippage": 0,
            },
            "sys_transaction_cost": {
                "enabled": True,
                "cn_stock_min_commission": float(min_commission),
                "stock_commission_multiplier": float(
                    stock_commission_multiplier
                ),
                "tax_multiplier": float(tax_multiplier),
            },
            "sys_progress": {
                "enabled": True,
                "show": bool(show_progress),
            },
        },
    }

    return run_func(
        config=config,
        init=init,
        before_trading=before_trading,
        open_auction=open_auction,
        after_trading=after_trading,
    )

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
LIB_DIR = PROJECT_ROOT / "lib"

LIB_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(PROJECT_ROOT))

if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

print("当前工作目录：", Path.cwd())
print("工具库目录：", LIB_DIR)

In [ ]:
#7.1.5
import pickle
import pandas as pd

TARGET_FILE = (
    PROJECT_ROOT
    / "data"
    / "strategy_inputs"
    / "07_oos_q5_target_weights_portable.pkl"
)

assert TARGET_FILE.exists(), f"未找到 RQAlpha 输入文件：{TARGET_FILE}"

with TARGET_FILE.open("rb") as file:
    target_payload = pickle.load(file)

assert target_payload["format"] == "rqalpha_target_weights_portable_v1"

rqalpha_q5_targets = pd.DataFrame(
    target_payload["weights"],
    index=pd.to_datetime(target_payload["dates"]),
    columns=target_payload["assets"],
    dtype=float,
)

print("===== 已读取跨版本 RQAlpha Q5 权重 =====")
print("形状：", rqalpha_q5_targets.shape)
print("日期：", rqalpha_q5_targets.index.min().date(), "至",
      rqalpha_q5_targets.index.max().date())

In [ ]:
# 07.2：RQAlpha 正式样本外回测
# 前提：已运行 07.0，已有 rqalpha_q5_targets 和 PROJECT_ROOT

RQALPHA_RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_q5_analyser.pkl"
)
RQALPHA_PLOT_DIR = (
    PROJECT_ROOT / "cache" / "rqalpha" / "07_oos_q5" / "plots"
)

print("===== RQAlpha 样本外 Q5 回测 =====")
print("阶段 1/3：检查 RQAlpha 原始结果缓存")

try:
    if RQALPHA_RAW_PATH.exists():
        rqalpha_result = pd.read_pickle(RQALPHA_RAW_PATH)
        print("缓存命中：直接读取既有 RQAlpha 原始结果。")
    else:
        print("阶段 2/3：运行 RQAlpha；界面将显示回测进度条。")

        rqalpha_result = rau.run_target_weight_backtest(
            target_weights=rqalpha_q5_targets,
            initial_cash=100_000_000,
            benchmark="000906.XSHG",
            analyser_output_file=RQALPHA_RAW_PATH,
            plot_output_path=RQALPHA_PLOT_DIR,
            matching_type="current_bar",
            min_commission=5.0,
            stock_commission_multiplier=0.375,
            tax_multiplier=1.0,
            price_limit=True,
            volume_limit=True,
            volume_percent=0.05,
            show_progress=True,
        )
        print("阶段 2/3：完成。")

    print("阶段 3/3：原始结果已保存，仅保留在 cache/，不会导出。")
    print("结果类型：", type(rqalpha_result))

    if isinstance(rqalpha_result, dict):
        print("可用结果字段：", sorted(rqalpha_result.keys()))

except Exception as exc:
    print("RQAlpha 回测未完成；原始研究缓存与因子结果未受影响。")
    raise exc

In [ ]:
import sys
import importlib

# 移除旧的工具库引用，再从文件重新导入
sys.modules.pop("rqalpha_strategy_utils", None)

rau = importlib.import_module("rqalpha_strategy_utils")

print("已重新导入工具库：", rau.__file__)

In [ ]:
# 07.2a：RQAlpha 短窗口冒烟测试
# 仅取前两个调仓事件，验证 RQAlpha 环境、目标权重和交易回调。

SMOKE_TARGETS = rqalpha_q5_targets.loc[
    rqalpha_q5_targets.index[:2]
].copy()

SMOKE_OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_q5_smoke_test.pkl"
)

SMOKE_PLOTS = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_q5_smoke_test_plots"
)

print("冒烟测试区间：",
      SMOKE_TARGETS.index.min().date(),
      "至",
      SMOKE_TARGETS.index.max().date())
print("目标股票列数：", SMOKE_TARGETS.shape[1])

rqalpha_smoke_result = rau.run_target_weight_backtest(
    target_weights=SMOKE_TARGETS,
    initial_cash=100_000_000,
    benchmark="000906.XSHG",
    analyser_output_file=SMOKE_OUTPUT,
    plot_output_path=SMOKE_PLOTS,
    matching_type="current_bar",
    min_commission=5.0,
    stock_commission_multiplier=0.375,
    tax_multiplier=1.0,
    price_limit=True,
    volume_limit=True,
    volume_percent=0.05,
    show_progress=True,
)

print("RQAlpha 冒烟测试完成。")
print("结果类型：", type(rqalpha_smoke_result))


In [ ]:
# 07.3.0：检查 RQAlpha 原始结果结构
import pandas as pd

print("原始 RQAlpha 结果文件：", RQALPHA_RAW_PATH)
print("文件是否存在：", RQALPHA_RAW_PATH.exists())

if "rqalpha_result" not in globals():
    rqalpha_result = pd.read_pickle(RQALPHA_RAW_PATH)

analysis = rqalpha_result["sys_analyser"]

print("\n===== sys_analyser 可用字段 =====")
print(sorted(analysis.keys()))

print("\n===== 各字段结构 =====")
for name, value in analysis.items():
    if isinstance(value, pd.DataFrame):
        print(
            name,
            "DataFrame",
            "shape=", value.shape,
            "columns=", list(value.columns)[:12],
        )
    elif isinstance(value, pd.Series):
        print(name, "Series", "length=", len(value))
    else:
        print(name, type(value))

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import pickle
PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

RQALPHA_RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_q5_analyser.pkl"
)

TARGET_FILE = (
    PROJECT_ROOT
    / "data"
    / "strategy_inputs"
    / "07_oos_q5_target_weights_portable.pkl"
)

print("已准备 RQAlpha 结果读取环境。")
print("原始结果：", RQALPHA_RAW_PATH)

In [ ]:
# 07.3.1：导出 RQAlpha Q5 样本外回测的安全研究产物
# RQAlpha 原始 analyser pkl 跨 pandas 版本不稳定；
# 当前回测已完成，优先使用内存中的正式结果。
assert "rqalpha_result" in globals(), (
    "当前 Kernel 中没有 rqalpha_result。请不要重启；"
    "若确实丢失内存，再单独处理可移植结果文件。"
)

raw_result = rqalpha_result
assert isinstance(raw_result, dict)
assert "sys_analyser" in raw_result

analysis = raw_result["sys_analyser"]

print("已直接使用当前 RQAlpha 回测内存结果。")
print("可用字段：", sorted(analysis.keys()))

TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. 安全绩效摘要
summary_rows = []
for metric, value in analysis["summary"].items():
    if isinstance(value, (dict, list, tuple)):
        value = json.dumps(value, ensure_ascii=False, default=str)
    summary_rows.append({"metric": metric, "value": value})

rqalpha_summary = pd.DataFrame(summary_rows)

# 2. 策略、基准、相对净值与回撤
portfolio = analysis["portfolio"].copy()
benchmark = analysis["benchmark_portfolio"].copy()

portfolio.index = pd.to_datetime(portfolio.index).normalize()
benchmark.index = pd.to_datetime(benchmark.index).normalize()

daily_nav = pd.DataFrame(
    {
        "strategy_nav": portfolio["unit_net_value"],
        "benchmark_nav": benchmark["unit_net_value"],
    }
).dropna()

daily_nav["relative_nav"] = (
    daily_nav["strategy_nav"] / daily_nav["benchmark_nav"]
)
daily_nav["strategy_drawdown"] = (
    daily_nav["strategy_nav"]
    / daily_nav["strategy_nav"].cummax()
    - 1.0
)
daily_nav["benchmark_drawdown"] = (
    daily_nav["benchmark_nav"]
    / daily_nav["benchmark_nav"].cummax()
    - 1.0
)

daily_nav_output = (
    daily_nav.reset_index()
    .rename(columns={"index": "date"})
)

# 3. 不含股票代码的成交与费用聚合审计
trades = analysis["trades"].copy()
trades["trade_date"] = pd.to_datetime(
    trades["trading_datetime"]
).dt.normalize()

trades["filled_notional"] = (
    trades["last_quantity"].abs()
    * trades["last_price"].abs()
)

trade_side = trades["side"].astype(str).str.upper()

execution_summary = pd.DataFrame(
    [
        {
            "trade_record_count": len(trades),
            "trade_day_count": trades["trade_date"].nunique(),
            "buy_filled_notional": trades.loc[
                trade_side.str.contains("BUY"),
                "filled_notional",
            ].sum(),
            "sell_filled_notional": trades.loc[
                trade_side.str.contains("SELL"),
                "filled_notional",
            ].sum(),
            "total_filled_notional": trades["filled_notional"].sum(),
            "total_commission": trades["commission"].sum(),
            "total_tax": trades["tax"].sum(),
            "total_transaction_cost": (
                trades["commission"].sum() + trades["tax"].sum()
            ),
        }
    ]
)

execution_daily = (
    trades.groupby("trade_date")
    .agg(
        trade_record_count=("order_id", "count"),
        filled_notional=("filled_notional", "sum"),
        commission=("commission", "sum"),
        tax=("tax", "sum"),
    )
    .reset_index()
)

# 4. 保存安全产物
summary_path = (
    TABLE_DIR / "07_rqalpha_oos_q5_performance_summary.csv"
)
nav_path = (
    TABLE_DIR / "07_rqalpha_oos_q5_daily_nav.csv"
)
execution_path = (
    TABLE_DIR / "07_rqalpha_oos_q5_execution_summary.csv"
)
execution_daily_path = (
    TABLE_DIR / "07_rqalpha_oos_q5_daily_execution.csv"
)
rules_path = (
    METADATA_DIR / "07_rqalpha_oos_q5_backtest_rules.json"
)
figure_path = (
    FIGURE_DIR / "07_rqalpha_oos_q5_performance.png"
)

rqalpha_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
daily_nav_output.to_csv(nav_path, index=False, encoding="utf-8-sig")
execution_summary.to_csv(execution_path, index=False, encoding="utf-8-sig")
execution_daily.to_csv(
    execution_daily_path,
    index=False,
    encoding="utf-8-sig",
)

rules = {
    "engine": "rqalpha_plus",
    "strategy": "integrated_profitability_structure_proxy_q5_long_only",
    "evaluation_scope": "out_of_sample_2023_2025_signals_with_final_exit",
    "benchmark": "000906.XSHG",
    "target_weight_input": str(TARGET_FILE),
    "raw_backtest_result": str(RQALPHA_RAW_PATH),
    "matching_type": "current_bar",
    "price_limit": True,
    "volume_limit": True,
    "volume_percent": 0.05,
    "minimum_commission": 5.0,
    "stock_commission_multiplier": 0.375,
    "tax_multiplier": 1.0,
    "rejected_order_audit_note": (
        "sys_analyser contains filled trades but not full rejected-order logs. "
        "Console showed final-day limit-down rejections; realized NAV reflects them."
    ),
}

with rules_path.open("w", encoding="utf-8") as file:
    json.dump(rules, file, ensure_ascii=False, indent=2)

# 5. 净值与回撤图
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(
    daily_nav.index,
    daily_nav["strategy_nav"],
    label="RQAlpha Q5 strategy",
)
axes[0].plot(
    daily_nav.index,
    daily_nav["benchmark_nav"],
    label="CSI 800",
)
axes[0].set_title("RQAlpha Out-of-Sample NAV")
axes[0].set_ylabel("Normalized NAV")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].fill_between(
    daily_nav.index,
    daily_nav["strategy_drawdown"],
    0,
    alpha=0.45,
    label="Strategy drawdown",
)
axes[1].plot(
    daily_nav.index,
    daily_nav["benchmark_drawdown"],
    label="CSI 800 drawdown",
)
axes[1].set_title("RQAlpha Drawdown")
axes[1].set_ylabel("Drawdown")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print("===== RQAlpha Q5 正式回测产物已保存 =====")
print("绩效摘要：", summary_path)
print("逐日净值：", nav_path)
print("执行摘要：", execution_path)
print("逐日成交聚合：", execution_daily_path)
print("规则元数据：", rules_path)
print("图表：", figure_path)

print("\n===== RQAlpha 汇总指标 =====")
display(rqalpha_summary)

print("\n===== 执行摘要 =====")
display(execution_summary)

In [ ]:
# 07.3.2：保存跨环境可读取的 RQAlpha 原始结果 pkl
PORTABLE_RESULT_FILE = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_q5_portable_result.pkl"
)

portable_result = {
    "format": "rqalpha_analyser_portable_v1",
    "summary": json.loads(
        json.dumps(
            analysis["summary"],
            ensure_ascii=False,
            default=str,
        )
    ),
    "portfolio": json.loads(
        analysis["portfolio"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    "benchmark_portfolio": json.loads(
        analysis["benchmark_portfolio"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    "stock_account": json.loads(
        analysis["stock_account"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    # 本地原始交易明细：含股票代码，不进入 artifacts 或 GitHub。
    "trades": json.loads(
        analysis["trades"].to_json(orient="records", date_format="iso")
    ),
}

with PORTABLE_RESULT_FILE.open("wb") as file:
    pickle.dump(portable_result, file, protocol=4)

print("已保存可移植 RQAlpha 结果：", PORTABLE_RESULT_FILE)
print("文件大小（MB）：", round(PORTABLE_RESULT_FILE.stat().st_size / 1024**2, 2))

In [ ]:
# 07.4.0：导出 Q1 与等权可投资券池的跨环境 RQAlpha 权重输入
from pathlib import Path
import pickle

import pandas as pd

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

SOURCE_FILE = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_targets.pkl"
)

INPUT_DIR = PROJECT_ROOT / "data" / "strategy_inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

source_bundle = pd.read_pickle(SOURCE_FILE)

required_keys = {
    "q1_long_control": "q1_event_targets",
    "equal_weight_eligible_universe": "equal_weight_event_targets",
}

missing_keys = [
    source_key
    for source_key in required_keys.values()
    if source_key not in source_bundle
]

if missing_keys:
    raise RuntimeError(
        "冻结的归因权重缓存字段与预期不一致。\n"
        "缺少：" + ", ".join(missing_keys) + "\n"
        "实际字段：" + ", ".join(sorted(source_bundle.keys()))
    )

for portfolio_name, source_key in required_keys.items():
    target_weights = source_bundle[source_key].copy()
    target_weights.index = pd.to_datetime(
        target_weights.index
    ).normalize()

    portable_payload = {
        "format": "rqalpha_target_weights_portable_v1",
        "portfolio_name": portfolio_name,
        "dates": target_weights.index.strftime("%Y-%m-%d").tolist(),
        "assets": target_weights.columns.astype(str).tolist(),
        "weights": (
            target_weights.fillna(0.0)
            .astype(float)
            .values
            .tolist()
        ),
    }

    output_file = (
        INPUT_DIR
        / "07_{}_target_weights_portable.pkl".format(portfolio_name)
    )

    with output_file.open("wb") as file:
        pickle.dump(portable_payload, file, protocol=4)

    print("已生成：", output_file)
    print(
        "  日期数：{}；股票列数：{}；日期范围：{} 至 {}".format(
            len(portable_payload["dates"]),
            len(portable_payload["assets"]),
            portable_payload["dates"][0],
            portable_payload["dates"][-1],
        )
    )

In [ ]:
# 07.4.1：读取并审计 RQAlpha 对照组合目标权重
from pathlib import Path
import pickle

import pandas as pd

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

CONTROL_INPUT_FILES = {
    "q1_long_control": (
        PROJECT_ROOT
        / "data"
        / "strategy_inputs"
        / "07_q1_long_control_target_weights_portable.pkl"
    ),
    "equal_weight_eligible_universe": (
        PROJECT_ROOT
        / "data"
        / "strategy_inputs"
        / "07_equal_weight_eligible_universe_target_weights_portable.pkl"
    ),
}

control_target_panels = {}
control_audit_rows = []

for portfolio_name, input_file in CONTROL_INPUT_FILES.items():
    assert input_file.exists(), f"未找到输入文件：{input_file}"

    with input_file.open("rb") as file:
        payload = pickle.load(file)

    assert payload["format"] == "rqalpha_target_weights_portable_v1"

    target_panel = pd.DataFrame(
        payload["weights"],
        index=pd.to_datetime(payload["dates"]),
        columns=payload["assets"],
        dtype=float,
    )

    control_target_panels[portfolio_name] = target_panel

    control_audit_rows.append(
        {
            "portfolio": portfolio_name,
            "event_date_count": len(target_panel),
            "asset_column_count": target_panel.shape[1],
            "first_event_date": target_panel.index.min().date(),
            "last_event_date": target_panel.index.max().date(),
            "mean_target_gross_weight": target_panel.sum(axis=1).mean(),
            "min_target_gross_weight": target_panel.sum(axis=1).min(),
            "max_target_gross_weight": target_panel.sum(axis=1).max(),
        }
    )

control_input_audit = pd.DataFrame(control_audit_rows)

print("===== RQAlpha 对照组合权重读取完成 =====")
display(control_input_audit)

In [ ]:
# 07.4.2：RQAlpha 正式样本外 Q1 对照回测
import sys
import importlib
import json
import pickle

LIB_DIR = PROJECT_ROOT / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

# RQAlpha 每次运行后会改写回调模块；必须重新导入，不使用 reload。
sys.modules.pop("rqalpha_strategy_utils", None)
rau = importlib.import_module("rqalpha_strategy_utils")

Q1_TARGETS = control_target_panels["q1_long_control"].copy()

Q1_ANALYSER_FILE = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_q1_analyser.pkl"
)

Q1_PLOT_DIR = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_q1_plots"
)

Q1_PORTABLE_RESULT_FILE = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_q1_portable_result.pkl"
)

print("===== RQAlpha 样本外 Q1 对照回测 =====")
print("目标日期：", Q1_TARGETS.index.min().date(), "至",
      Q1_TARGETS.index.max().date())
print("每 20 个交易日会输出一次进度。")

q1_rqalpha_result = rau.run_target_weight_backtest(
    target_weights=Q1_TARGETS,
    initial_cash=100_000_000,
    benchmark="000906.XSHG",
    analyser_output_file=Q1_ANALYSER_FILE,
    plot_output_path=Q1_PLOT_DIR,
    matching_type="current_bar",
    min_commission=5.0,
    stock_commission_multiplier=0.375,
    tax_multiplier=1.0,
    price_limit=True,
    volume_limit=True,
    volume_percent=0.05,
    show_progress=True,
    progress_interval=20,
)

assert isinstance(q1_rqalpha_result, dict)
q1_analysis = q1_rqalpha_result["sys_analyser"]

# 立即保存跨 Python / pandas 版本可读取的本地结果。
q1_portable_result = {
    "format": "rqalpha_analyser_portable_v1",
    "portfolio_name": "q1_long_control",
    "summary": json.loads(
        json.dumps(q1_analysis["summary"], ensure_ascii=False, default=str)
    ),
    "portfolio": json.loads(
        q1_analysis["portfolio"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    "benchmark_portfolio": json.loads(
        q1_analysis["benchmark_portfolio"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    "stock_account": json.loads(
        q1_analysis["stock_account"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    "trades": json.loads(
        q1_analysis["trades"].to_json(
            orient="records",
            date_format="iso",
        )
    ),
}

with Q1_PORTABLE_RESULT_FILE.open("wb") as file:
    pickle.dump(q1_portable_result, file, protocol=4)

print("===== Q1 RQAlpha 回测完成 =====")
print("可移植结果：", Q1_PORTABLE_RESULT_FILE)
print("可用字段：", sorted(q1_analysis.keys()))
print("策略累计收益：", q1_analysis["summary"].get("total_returns"))
print("策略年化收益：", q1_analysis["summary"].get("annualized_returns"))

In [ ]:
# 07.4.3：RQAlpha 正式样本外等权可投资券池回测
import sys
import importlib
import json
import pickle

LIB_DIR = PROJECT_ROOT / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

sys.modules.pop("rqalpha_strategy_utils", None)
rau = importlib.import_module("rqalpha_strategy_utils")

UNIVERSE_TARGETS = control_target_panels[
    "equal_weight_eligible_universe"
].copy()

UNIVERSE_ANALYSER_FILE = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_equal_weight_eligible_universe_analyser.pkl"
)

UNIVERSE_PLOT_DIR = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_equal_weight_eligible_universe_plots"
)

UNIVERSE_PORTABLE_RESULT_FILE = (
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "07_rqalpha_oos_equal_weight_eligible_universe_portable_result.pkl"
)

print("===== RQAlpha 样本外等权可投资券池回测 =====")
print("目标日期：", UNIVERSE_TARGETS.index.min().date(), "至",
      UNIVERSE_TARGETS.index.max().date())
print("每 20 个交易日会输出一次进度。")

universe_rqalpha_result = rau.run_target_weight_backtest(
    target_weights=UNIVERSE_TARGETS,
    initial_cash=100_000_000,
    benchmark="000906.XSHG",
    analyser_output_file=UNIVERSE_ANALYSER_FILE,
    plot_output_path=UNIVERSE_PLOT_DIR,
    matching_type="current_bar",
    min_commission=5.0,
    stock_commission_multiplier=0.375,
    tax_multiplier=1.0,
    price_limit=True,
    volume_limit=True,
    volume_percent=0.05,
    show_progress=True,
    progress_interval=20,
)

assert isinstance(universe_rqalpha_result, dict)
universe_analysis = universe_rqalpha_result["sys_analyser"]

universe_portable_result = {
    "format": "rqalpha_analyser_portable_v1",
    "portfolio_name": "equal_weight_eligible_universe",
    "summary": json.loads(
        json.dumps(
            universe_analysis["summary"],
            ensure_ascii=False,
            default=str,
        )
    ),
    "portfolio": json.loads(
        universe_analysis["portfolio"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    "benchmark_portfolio": json.loads(
        universe_analysis["benchmark_portfolio"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    "stock_account": json.loads(
        universe_analysis["stock_account"]
        .reset_index()
        .to_json(orient="split", date_format="iso")
    ),
    "trades": json.loads(
        universe_analysis["trades"].to_json(
            orient="records",
            date_format="iso",
        )
    ),
}

with UNIVERSE_PORTABLE_RESULT_FILE.open("wb") as file:
    pickle.dump(universe_portable_result, file, protocol=4)

print("===== 等权可投资券池回测完成 =====")
print("可移植结果：", UNIVERSE_PORTABLE_RESULT_FILE)
print("策略累计收益：",
      universe_analysis["summary"].get("total_returns"))
print("策略年化收益：",
      universe_analysis["summary"].get("annualized_returns"))